# SciScape Quickstart

This notebook demonstrates the core SciScape workflow:
1. Query OpenAlex for papers
2. Build citation edges (DC, BC, CC)
3. Combine multi-layer edges with consensus weighting
4. Run hierarchical Leiden clustering
5. Visualize results

In [ ]:
# Install if needed:
# !pip install sciscape[openalex,viz]

## 1. Query OpenAlex

In [ ]:
from sciscape.openalex import OpenAlexClient

client = OpenAlexClient(email="your@email.com")  # polite pool
works = client.search_works(
    "network science community detection",
    filters={"publication_year": "2020-2024"},
    max_results=500,
)
print(f"Fetched {len(works)} papers")

## 2. Build Citation Edges

In [ ]:
from sciscape.openalex.edges import build_citation_edges

edge_tables = build_citation_edges(works, bc=True, cc=True)

for name, df in edge_tables.items():
    print(f"{name}: {df.height:,} edges")

## 3. Combine with Consensus Weighting

In [ ]:
from sciscape.linkage.combine import combine_edge_layers

combined = combine_edge_layers(
    edge_tables,
    strategy="consensus",  # weight × n_layers
    top_k="auto",          # adaptive sqrt(n)
    gcc=True,               # giant connected component only
)
print(f"Combined: {combined.height:,} edges")
combined.head(5)

## 4. Hierarchical Clustering

In [ ]:
from sciscape.clustering.hierarchical import build_hierarchy

result = build_hierarchy(
    edges=combined,
    n_levels=2,
    targets={"nano": 3.0, "micro": 10.0},
    min_sizes={"nano": 10, "micro": 30},
)

for level in result.levels:
    print(f"{level.name}: {level.n_clusters} clusters, "
          f"max={level.max_pct}%, γ={level.gamma:.2e}")

## 5. Consensus Visualization

In [ ]:
from sciscape.visualization.consensus import compute_consensus_stats, format_consensus_report

stats = compute_consensus_stats(edge_tables)
print(format_consensus_report(stats))

## 6. Auto-Gamma Search (standalone)

If you want to find the optimal γ for a specific edge set:

In [ ]:
from sciscape.clustering.auto_gamma import find_gamma

gamma_result = find_gamma(
    combined,
    target_max_pct=3.0,  # largest cluster < 3% of total
    min_size=10,
)
print(f"Best γ={gamma_result.gamma:.2e}")
print(f"  {gamma_result.n_clusters} clusters, max={gamma_result.max_pct}%")
print(f"  Probed {len(gamma_result.probes)} gamma values")

## 7. One-Line Pipeline

For the full end-to-end pipeline in one call:

In [ ]:
from sciscape.openalex import run_openalex_pipeline, OpenAlexPipelineConfig

config = OpenAlexPipelineConfig(
    query="machine learning drug discovery",
    max_works=300,
    email="your@email.com",
    edge_types=["dc", "bc"],
    run_landscape=True,
)
result = run_openalex_pipeline(config)
print(f"Done: {result.n_works} works, {sum(result.n_edges.values())} edges")
print(f"Landscape: {result.landscape_dir}")